In [2]:
# ======================================
# Full pipeline: higher precision via
# (1) SVM tuning for precision-weighted F0.5
# (2) Out-of-fold probability threshold selection
# (3) Train final model + save model + save threshold
# ======================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    make_scorer,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    classification_report,
)
from sklearn.svm import SVC
from imblearn.pipeline import Pipeline as ImbPipeline


# --------------------------
# 1) Load & preprocess data
# --------------------------
df = pd.read_excel("input/DB_Abbott_OCT2025.xlsx")

df1 = df[
    [
        "Pathogen Discovery",
        "Pathogens confirmed (acute case)",
        "Sex",
        "Age",
        "Year",
        "Month",
        "Site",
        "Ocupation",
        "Temperature",
        "Shaking chills",
        "Headache",
        "discomfort",
        "irritability",
        "confusion",
        "fatigue",
        "appetite loss",
        "chest pain",
        "back pain",
        "Abdominal pain",
        "Bone-ache",
        "Muscle pain",
        "Retro orbital pain",
        "Weakness",
        "sore throat",
        "Cough",
        "Diarrhea",
        "nausea",
        "Vomit",
        "Rhinorrhea",
        "Red eyes",
        "Skin rash",
        "Jaundice",
        "Echymosis /Hemorrhages",
        "Respiratory distress",
        "Dizziness",
        "Petechiae",
        "Anosmia or ageusia",
        "hospitalized",
        "transfusion",
        "SMOKER",
    ]
].copy()

# Target
df1.loc[:, "Pathogen Present"] = df1["Pathogen Discovery"].apply(lambda x: 0 if x == "NO" else 1)
df1.loc[:, "Pathogen Present"] = df1["Pathogen Present"].where(
    df1["Pathogens confirmed (acute case)"] == "UNDETERMINED", 1
)

# Drop columns (your original drops)
df1 = df1.drop(
    columns=[
        "discomfort",
        "irritability",
        "confusion",
        "fatigue",
        "appetite loss",
        "chest pain",
        "back pain",
        "nausea",
        "Vomit",
        "Jaundice",
        "Dizziness",
        "Petechiae",
        "Anosmia or ageusia",
        "transfusion",
        "SMOKER",
    ]
)

# Temperature cleaning
df1["Temperature"] = (
    df1["Temperature"]
    .astype(str)
    .str.strip()
    .str.replace(r"[^\d\.\-]", "", regex=True)
)
df1["Temperature"] = pd.to_numeric(df1["Temperature"], errors="coerce")
df1["Temperature"] = df1["Temperature"].fillna(df1["Temperature"].median())

# Normalize YES/NO fields (same as your code)
for col in ["Shaking chills", "Headache", "Abdominal pain", "Bone-ache", "Muscle pain", "Weakness"]:
    df1.loc[:, col] = df1[col].where(df1[col].eq("YES"), "NO")

for col in [
    "sore throat",
    "Retro orbital pain",
    "Cough",
    "Diarrhea",
    "Rhinorrhea",
    "Red eyes",
    "Skin rash",
    "Echymosis /Hemorrhages",
    "Respiratory distress",
    "hospitalized",
]:
    df1.loc[:, col] = df1[col].where(df1[col].eq("YES"), "NO")

# X/y
X = df1.drop(columns=["Pathogen Discovery", "Pathogens confirmed (acute case)", "Pathogen Present"])
y = df1["Pathogen Present"].astype(int)


# --------------------------
# 2) Preprocessor
# --------------------------
categorical_features = [
    "Sex",
    "Month",
    "sore throat",
    "Retro orbital pain",
    "Cough",
    "Diarrhea",
    "Shaking chills",
    "Headache",
    "Abdominal pain",
    "Bone-ache",
    "Muscle pain",
    "Weakness",
    "Rhinorrhea",
    "Red eyes",
    "Skin rash",
    "Echymosis /Hemorrhages",
    "Respiratory distress",
    "hospitalized",
    "Site",
    "Ocupation",
]
numerical_features = ["Age", "Year", "Temperature"]

categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
numerical_transformer = MinMaxScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("encoder", categorical_transformer, categorical_features),
        ("scaler", numerical_transformer, numerical_features),
    ]
)


# --------------------------
# 3) CV + scoring summary
# --------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_summary = {
    "accuracy": make_scorer(accuracy_score),
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "f0_5": make_scorer(fbeta_score, beta=0.5, zero_division=0),  # for reporting
}

# Custom scorer for GridSearchCV (must be callable or valid string)
f0_5_scorer = make_scorer(fbeta_score, beta=0.5, zero_division=0)


# --------------------------
# 4) SVM pipeline + GridSearch
# --------------------------
svm_pipe = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("feature_selection", SelectKBest(score_func=f_classif, k=30)),
        (
            "classifier",
            SVC(
                kernel="rbf",
                probability=True,
                random_state=42,
                class_weight="balanced",
            ),
        ),
    ]
)

param_grid = {
    "feature_selection__k": [20, 30, 50, "all"],
    "classifier__C": [0.1, 1, 3, 10, 30, 100],
    "classifier__gamma": ["scale", 0.01, 0.03, 0.1, 0.3, 1.0],
}

grid = GridSearchCV(
    estimator=svm_pipe,
    param_grid=param_grid,
    scoring=f0_5_scorer,   # ✅ FIX: pass callable scorer, not "f0_5"
    cv=cv,
    n_jobs=-1,
    refit=True,
)

grid.fit(X, y)
best_model = grid.best_estimator_

print("\n✅ Best SVM params:", grid.best_params_)
print("✅ Best CV score (F0.5):", grid.best_score_)

# Report tuned model with default threshold (0.5)
scores = cross_validate(best_model, X, y, cv=cv, scoring=scoring_summary, n_jobs=-1)
print("\n📊 Tuned SVM (default threshold) CV means:")
for k, v in scores.items():
    if k.startswith("test_"):
        print(f"{k.replace('test_',''):>18}: {np.mean(v):.4f} ± {np.std(v):.4f}")


# --------------------------
# 5) Threshold selection to increase precision
# --------------------------
# Out-of-fold probabilities
proba_oof = cross_val_predict(best_model, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

prec, rec, thr = precision_recall_curve(y, proba_oof)

# Strategy A: maximize precision subject to minimum recall
MIN_RECALL = 0.50  # <-- adjust this (lower -> higher precision usually, but more FN)

valid = rec[:-1] >= MIN_RECALL  # thr aligns with prec[:-1], rec[:-1]
if np.any(valid):
    best_idx = np.argmax(np.where(valid, prec[:-1], -1))
else:
    # fallback: best F0.5 threshold
    f0_5_vals = (1 + 0.5**2) * (prec[:-1] * rec[:-1]) / ((0.5**2 * prec[:-1]) + rec[:-1] + 1e-12)
    best_idx = int(np.nanargmax(f0_5_vals))

best_threshold = float(thr[best_idx])

print("\n🎯 Chosen threshold:", best_threshold)
print(f"   OOF Precision @thr: {prec[best_idx]:.4f}")
print(f"   OOF Recall    @thr: {rec[best_idx]:.4f}")

# Evaluate OOF predictions at chosen threshold
y_pred_oof = (proba_oof >= best_threshold).astype(int)

print("\n📌 OOF classification report @ chosen threshold:")
print(classification_report(y, y_pred_oof, digits=4, zero_division=0))

print("OOF precision:", precision_score(y, y_pred_oof, zero_division=0))
print("OOF recall   :", recall_score(y, y_pred_oof, zero_division=0))
print("OOF f1       :", f1_score(y, y_pred_oof, zero_division=0))
print("OOF f0.5     :", fbeta_score(y, y_pred_oof, beta=0.5, zero_division=0))


# --------------------------
# 6) Train final model on full data + save model + threshold
# --------------------------
final_model = best_model.fit(X, y)

model_path = "best_model_precision_SVM_RBF.pkl"
joblib.dump(final_model, model_path)

threshold_info = {
    "threshold": best_threshold,
    "min_recall_constraint": MIN_RECALL,
    "grid_best_params": grid.best_params_,
    "selection_metric": "F0.5",
}
joblib.dump(threshold_info, "best_model_precision_SVM_RBF_threshold.pkl")

print(f"\n💾 Saved model to: {model_path}")
print("💾 Saved threshold info to: best_model_precision_SVM_RBF_threshold.pkl")


# --------------------------
# 7) Example usage (prediction)
# --------------------------
# loaded_model = joblib.load("best_model_precision_SVM_RBF.pkl")
# thr_info = joblib.load("best_model_precision_SVM_RBF_threshold.pkl")
# t = thr_info["threshold"]
# proba_new = loaded_model.predict_proba(X_new)[:, 1]
# y_pred_new = (proba_new >= t).astype(int)



✅ Best SVM params: {'classifier__C': 1, 'classifier__gamma': 0.3, 'feature_selection__k': 'all'}
✅ Best CV score (F0.5): 0.500606921565803

📊 Tuned SVM (default threshold) CV means:
          accuracy: 0.7010 ± 0.0134
 balanced_accuracy: 0.6857 ± 0.0144
         precision: 0.4734 ± 0.0174
            recall: 0.6510 ± 0.0241
                f1: 0.5480 ± 0.0175
              f0_5: 0.5006 ± 0.0172

🎯 Chosen threshold: 0.4058144789968937
   OOF Precision @thr: 0.5141
   OOF Recall    @thr: 0.5001

📌 OOF classification report @ chosen threshold:
              precision    recall  f1-score   support

           0     0.8092    0.8177    0.8134      8854
           1     0.5141    0.5001    0.5071      3415

    accuracy                         0.7293     12269
   macro avg     0.6617    0.6589    0.6602     12269
weighted avg     0.7271    0.7293    0.7282     12269

OOF precision: 0.5141481035520771
OOF recall   : 0.5001464128843338
OOF f1       : 0.5070506160011875
OOF f0.5     : 0.511285

In [3]:
import pandas as pd
import numpy as np
import joblib

# ======================================
# 1) Load new dataset
# ======================================
new_df = pd.read_excel("input/seleccion-nuevas-muestras-nov-dic2025.xlsx")

# Keep ID/reference columns (safe if missing)
def safe_col(df, col):
    return df[col] if col in df.columns else pd.Series([np.nan] * len(df))

ids = safe_col(new_df, "ID")
pathogens_confirmed = safe_col(new_df, "Pathogens confirmed (acute case)")
metagenomics_seq = safe_col(new_df, "Metagenomics Seq")

# Predictor columns (must match training inputs)
predictor_cols = [
    "Sex","Age","Year","Month","Site","Ocupation","Temperature",
    "Shaking chills","Headache","Abdominal pain","Bone-ache","Muscle pain","Weakness",
    "sore throat","Retro orbital pain","Cough","Diarrhea","Rhinorrhea","Red eyes",
    "Skin rash","Echymosis /Hemorrhages","Respiratory distress","hospitalized"
]

missing = [c for c in predictor_cols if c not in new_df.columns]
if missing:
    raise ValueError(f"❌ Missing required columns in new_df: {missing}")

X_new = new_df[predictor_cols].copy()

# ======================================
# 2) Clean / normalize exactly like training
# ======================================

# Temperature cleaning
X_new["Temperature"] = (
    X_new["Temperature"]
    .astype(str)
    .str.strip()
    .str.replace(r"[^\d\.\-]", "", regex=True)
)
X_new["Temperature"] = pd.to_numeric(X_new["Temperature"], errors="coerce")

# Numeric safety
for num_col in ["Age", "Year"]:
    X_new[num_col] = pd.to_numeric(X_new[num_col], errors="coerce")

# Fill numeric missing with medians (per new dataset)
for num_col in ["Age", "Year", "Temperature"]:
    X_new[num_col] = X_new[num_col].fillna(X_new[num_col].median())

# Normalize YES/NO symptom variables
for col in ["Shaking chills","Headache","Abdominal pain","Bone-ache","Muscle pain","Weakness"]:
    X_new.loc[:, col] = X_new[col].where(X_new[col].eq("YES"), "NO")

for col in [
    "sore throat","Retro orbital pain","Cough","Diarrhea",
    "Rhinorrhea","Red eyes","Skin rash","Echymosis /Hemorrhages",
    "Respiratory distress","hospitalized"
]:
    X_new.loc[:, col] = X_new[col].where(X_new[col].eq("YES"), "NO")

# Strip whitespace from object columns
for col in X_new.columns:
    if X_new[col].dtype == object:
        X_new[col] = X_new[col].astype(str).str.strip()

print("✅ Missing values before prediction:")
print(X_new.isna().sum())

# ======================================
# 3) Load the trained model + threshold
# ======================================
model = joblib.load("best_model_precision_SVM_RBF.pkl")
thr_info = joblib.load("best_model_precision_SVM_RBF_threshold.pkl")
threshold = float(thr_info["threshold"])

print("\n✅ Loaded model + threshold")
print("Threshold:", threshold)
print("Grid best params:", thr_info.get("grid_best_params"))

# ======================================
# 4) Predict probabilities + apply threshold
# ======================================
if hasattr(model, "predict_proba"):
    prob_pos = model.predict_proba(X_new)[:, 1]
else:
    # If ever not available, fallback (should not happen with SVC(probability=True))
    scores = model.decision_function(X_new)
    prob_pos = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)

pred_label = (prob_pos >= threshold).astype(int)

# ======================================
# 5) Build results table
# ======================================
results = pd.DataFrame({
    "ID": ids,
    "Pathogens confirmed (acute case)": pathogens_confirmed,
    "Metagenomics Seq": metagenomics_seq,
    "Predicted_Pathogen_Present": pred_label,
    "Predicted_Probability": prob_pos,
    "Threshold_Used": threshold
})

# Optional context columns to include in the output
context_cols = ["Temperature", "Cough", "Headache", "Site", "Month"]
context_cols = [c for c in context_cols if c in new_df.columns]

# Bring context from X_new (cleaned) or new_df
# (Temperature is cleaned in X_new, so pull it from X_new)
context_from_xnew = [c for c in context_cols if c in X_new.columns]
context_from_newdf = [c for c in context_cols if c not in X_new.columns and c in new_df.columns]

if context_from_xnew:
    results = results.merge(pd.concat([ids.rename("ID"), X_new[context_from_xnew]], axis=1), on="ID", how="left")
if context_from_newdf:
    results = results.merge(new_df[["ID"] + context_from_newdf], on="ID", how="left")

# ======================================
# 6) Sort + save
# ======================================
results = results.sort_values(by="Predicted_Probability", ascending=False)

out_path = "predicted_pathogen_nov-dic_precision_model.xlsx"
results.to_excel(out_path, index=False)

print(f"\n✅ Prediction file saved as: {out_path}")

# Preview top 20
print("\n🔎 Top 20 highest-probability samples:")
display(results.head(20))


✅ Missing values before prediction:
Sex                       0
Age                       0
Year                      0
Month                     0
Site                      0
Ocupation                 0
Temperature               0
Shaking chills            0
Headache                  0
Abdominal pain            0
Bone-ache                 0
Muscle pain               0
Weakness                  0
sore throat               0
Retro orbital pain        0
Cough                     0
Diarrhea                  0
Rhinorrhea                0
Red eyes                  0
Skin rash                 0
Echymosis /Hemorrhages    0
Respiratory distress      0
hospitalized              0
dtype: int64

✅ Loaded model + threshold
Threshold: 0.4058144789968937
Grid best params: {'classifier__C': 1, 'classifier__gamma': 0.3, 'feature_selection__k': 'all'}

✅ Prediction file saved as: predicted_pathogen_nov-dic_precision_model.xlsx

🔎 Top 20 highest-probability samples:


,ID,Pathogens confirmed (acute case),Metagenomics Seq,Predicted_Pathogen_Present,Predicted_Probability,Threshold_Used,Temperature,Cough,Headache,Site,Month
81,0402117,UNDETERMINED,NT,1,0.751181,0.405814,37.0,NO,YES,APARTADÓ,DIC
151,LET-3361,UNDETERMINED,NT,1,0.707552,0.405814,40.0,NO,YES,LETICIA,DIC
138,LET-3348,UNDETERMINED,NT,1,0.692401,0.405814,37.5,NO,YES,LETICIA,DIC
150,LET-3360,DENV,NT,1,0.678180,0.405814,39.0,NO,YES,LETICIA,DIC
77,0402113,MALA,NT,1,0.677904,0.405814,37.5,NO,YES,APARTADÓ,DIC
154,LET-3364,MALA,NT,1,0.667659,0.405814,39.0,NO,YES,LETICIA,DIC
145,LET-3355,MALA,NT,1,0.660949,0.405814,39.9,NO,YES,LETICIA,DIC
75,0402111,DENV,NT,1,0.653082,0.405814,37.0,NO,YES,APARTADÓ,DIC
51,0402087,UNDETERMINED,NT,1,0.647927,0.405814,38.0,NO,YES,APARTADÓ,NOV
115,LET-3325,MALA,NT,1,0.628299,0.405814,40.0,NO,YES,LETICIA,NOV
